In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import time
import torch
import random
import numpy as np
from glob import glob
from pathlib import Path
from src.utilities.report import Report
from src.service.fragment.net import Net
from src.utilities.score_mapper import ScoreMapper
from src.utilities.load_dataset import load_dataset
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.service.stitching.generate_networks import generate_networks
from src.utilities.dataloader_generator import generate_dataloader

In [5]:
netsFiles = sorted(glob('../_results_without_finetune/fragments/net*'))
nets = []
for index, netsFile in enumerate(netsFiles):
    fragmentFiles = sorted(glob(str(Path(netsFile)/'fragment*.onnx')))
    onnxFragments = []
    for fragmentFile in fragmentFiles:
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
    net1 = Net(onnxFragments, index)
    nets.append(net1)

In [6]:
random.seed(51)
np.random.seed(24)
torch.manual_seed(77)

K = 5
STITCH_BATCH_SIZE = 32 # todo study the effect
MAX_DEPTH = 16
THRESOULD = 0
TOTAL_THRESOULD = 0.5

RESULT_NAME = f"{int(time.time())}_result_BS_{STITCH_BATCH_SIZE}_MD_{MAX_DEPTH}_T_{THRESOULD}_TT_{TOTAL_THRESOULD}_K_{K}"

EVAL_BATCH_SIZE = 64

train_datalader = generate_dataloader(load_dataset(), batch_size=STITCH_BATCH_SIZE)
test_datalader = generate_dataloader(load_dataset("test"), batch_size=EVAL_BATCH_SIZE)
data_score, _ = next(iter(train_datalader))
data_score = data_score.numpy()
print(data_score.shape)

(32, 3, 224, 224)


In [7]:
# range(1)
k = 0
if os.path.exists(f'../_results_without_finetune/{RESULT_NAME}.txt'):
    with open(f'../_results_without_finetune/{RESULT_NAME}.txt', 'r') as f:
        k = len(f.read().split('\n'))
        print(k)

In [8]:
scoreMapper = ScoreMapper(nets, data_score)
with Report(EVAL_BATCH_SIZE, f'../_results_without_finetune/{RESULT_NAME}.txt', 'a') as report:
    generator = generate_networks(nets, scoreMapper, data_score,
                                  threshold=THRESOULD, totalThreshold=TOTAL_THRESOULD,
                                  maxDepth=MAX_DEPTH, sample=False, K=K)
    for i, (score, net) in enumerate(generator):
        try:
            netname = f"../_results_without_finetune/{RESULT_NAME}/net{k:03}"
            report.evaluate(nets, net, netname, score, test_datalader)
            net.save(netname)
            k += 1
        except Exception as e:
            print('ERROR', e)
            pass

Exception in thread Thread-19 (net_scores):
Traceback (most recent call last):
  File "/home/shafigh/Desktop/projects/stitchnet/.stitchnet/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 419, in __init__
    self._create_inference_session(providers, provider_options, disabled_optimizers)
  File "/home/shafigh/Desktop/projects/stitchnet/.stitchnet/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py", line 483, in _create_inference_session
    sess.initialize_session(providers, provider_options, disabled_optimizers)
RuntimeError: /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:121 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:121 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:114 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] CUDA failure 900: operation not permitted when stream is capturing ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/cuda_execution_provider.cc ; line=248 ; expr=cudaDeviceSynchronize(); 

 when using ['CUDAExecutionProvider', 'CPUExecutionPr

2024-04-23 17:19:16.592975077 [E:onnxruntime:Default, cuda_call.cc:116 CudaCall] CUBLAS failure 13: CUBLAS_STATUS_EXECUTION_FAILED ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/math/gemm.cc ; line=121 ; expr=cublasGemmHelper( GetCublasHandle(ctx), CUBLAS_OP_N, CUBLAS_OP_N, N, M, 1, &one, b_data, N, GetConstOnes<CudaT>(M, Stream(ctx)), 1, &zero, out_data, N, device_prop); 
Exception in thread Thread-21 (net_scores):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
2024-04-23 17:19:16.593133931 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running Gemm node. Name:'/classifier/classifier.3/Gemm' Status Message: CUBLAS failure 13: CUBLAS_STATUS_EXECUTION_FAILED ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/math/gemm.cc ; line=121 ; expr=cublasGemmHelper( GetCublasHandle(ctx), CUBLAS_OP_N, CUBLAS_OP_N, N,

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:121 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:114 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] CUDA failure 900: operation not permitted when stream is capturing ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/cuda_execution_provider.cc ; line=248 ; expr=cudaDeviceSynchronize(); 

 when using ['CUDAExecutionProvider', 'CPUExecutionPr

Exception in thread 2024-04-23 17:19:24.704710204 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.28/Conv' Status Message: CUDA error cudaErrorStreamCaptureUnsupported:operation not permitted when stream is capturing
Thread-26 (net_scores):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/shafigh/Desktop/projects/stitchnet/.stitchnet/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/shafigh/Desktop/projects/stitchnet/src/utilities/score_mapper.py", line 18, in net_scores
    nextscores = net2.get_scores(x1,data,self.scoring_method)
  File "/home/shafigh/Desktop/projects/stitchnet/src/service/fragment/net.py", line 212, in get_s

...

100%|██████████| 158/158 [00:20<00:00,  7.69it/s]


accuracy 0.0
Node Init Time Elapsed 0.0010242462158203125
Tensor Init Time Elapsed 0.0018227100372314453
IO Tensor Init Time Elapsed 0.0001919269561767578
Constant Search Time Elapsed 2.956390380859375e-05
Update Nodes Tensors  Time Elapsed 0.00032520294189453125
{'Conv': 0.0001049041748046875, 'HardSwish': 2.765655517578125e-05, 'Relu': 2.2411346435546875e-05, 'GlobalAveragePool': 1.4781951904296875e-05, 'HardSigmoid': 1.1682510375976562e-05, 'Mul': 2.5272369384765625e-05, 'Add': 1.3589859008789062e-05, 'Flatten': 8.58306884765625e-06, 'Gemm': 4.291534423828125e-06}
saving to ../_results_without_finetune/1713880127_result_BS_32_MD_16_T_0_TT_0.5_K_5/net179.onnx
totalscore before thresholding of 0.5: 0.5475325114309995


100%|██████████| 158/158 [00:20<00:00,  7.71it/s]


accuracy 0.0
Node Init Time Elapsed 0.0010302066802978516
Tensor Init Time Elapsed 0.0018138885498046875
IO Tensor Init Time Elapsed 0.0002079010009765625
Constant Search Time Elapsed 2.9087066650390625e-05
Update Nodes Tensors  Time Elapsed 0.00032448768615722656
{'Conv': 0.00010919570922851562, 'HardSwish': 2.765655517578125e-05, 'Relu': 2.2649765014648438e-05, 'GlobalAveragePool': 1.5020370483398438e-05, 'HardSigmoid': 1.33514404296875e-05, 'Mul': 2.5272369384765625e-05, 'Add': 1.4066696166992188e-05, 'Flatten': 9.298324584960938e-06, 'Gemm': 4.291534423828125e-06}
saving to ../_results_without_finetune/1713880127_result_BS_32_MD_16_T_0_TT_0.5_K_5/net180.onnx
totalscore before thresholding of 0.5: 0.5410098450306111


  5%|▌         | 8/158 [00:01<00:21,  7.00it/s]